In [19]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm

import numpy as np
import pandas as pd
import ast
import glob
import pickle
import dask
import os
import itertools


#from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm

from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed
#from tqdm import tqdm

import dask
import dask.dataframe as dd
from dask.distributed import Client
from dask.diagnostics import ProgressBar

#client = Client(n_workers=20, memory_limit="10GB", interface='lo')
from concurrent.futures import ThreadPoolExecutor

import dask_ml.cluster as dask_cluster

from pprint import pprint
import os

pd.set_option('display.max_columns', None)

### Generate Fixed Window Estimates of $wsize \in \{2,3,4,\dotsc,13,14\}$

In [57]:
def RollingOLS_by_fips(df, fips, window_size):
    input_df = df.copy()
    input_df = input_df[input_df["fips"]==fips]
    input_df = input_df.sort_values(by="days_from_start")
    #display(df)
    print("Generating betas for fips={}, window_size={}".format(fips, window_size))
    beta = RollingOLS(endog=input_df["log_rolled_cases"], exog=sm.add_constant(input_df["days_from_start"]), window=window_size).fit().params
    beta = beta.rename(columns={"days_from_start":"beta_wsize={}".format(window_size)})
    return (fips, window_size, pd.DataFrame(beta["beta_wsize={}".format(window_size)]))

In [ ]:
benchmark_TLGRF_dataset = dd.read_csv("../benchmark_TLGRF_dataset.csv", assume_missing=True).compute()
benchmark_TLGRF_dataset["date"] = pd.to_datetime(benchmark_TLGRF_dataset["date"])

df = benchmark_TLGRF_dataset.copy()
window_sizes = range(2,15)
fips_list = df["fips"].unique()

In [79]:
#window_sizes = [2,3,4]
#fips_list = [1001,1003,99999]

with tqdm(total=len(window_sizes) * len(fips_list), desc="Processing") as pbar:
    def update_progress(*args):
        pbar.update()
    
    beta_results = Parallel(n_jobs=-1)(delayed(RollingOLS_by_fips)(df, fips, window_size) for fips in fips_list for window_size in window_sizes)
    pbar.close()

Generating betas for fips=1003, window_size=2
Generating betas for fips=1003, window_size=3
Generating betas for fips=1001, window_size=4
Generating betas for fips=99999, window_size=3
Generating betas for fips=99999, window_size=4
Generating betas for fips=1001, window_size=3
Generating betas for fips=1001, window_size=2
Generating betas for fips=1003, window_size=4
Generating betas for fips=99999, window_size=2


/home/zwang937/anaconda3/lib/python3.7/site-packages/statsmodels/regression/rolling.py:255: RuntimeWarning: divide by zero encountered in double_scalars
  s2 = ssr / (nobs - tot_params)


In [92]:
beta_result_dict = {window_size:[] for window_size in window_sizes}
for result in beta_results:
    fips, window_size, beta_df = result
    beta_result_dict[window_size].append(beta_df)

concatenated_beta_result_dict = {}
for window_size, beta_df_list in beta_result_dict.items():
    concatenated_beta_result_dict[window_size] = pd.concat(beta_df_list)

beta_df_big = pd.DataFrame()
for window_size, big_beta_df in concatenated_beta_result_dict.items():
    beta_col_name = big_beta_df.columns[0]
    beta_df_big[beta_col_name] = big_beta_df

updated_df = pd.merge(df, beta_df_big, left_index=True, right_index=True, how="outer").sort_values(by=["fips", 'days_from_start'])

updated_df.to_csv("TLGRF_w_Fixed_Windows.csv", index=False)

,fips,county,state,date,days_from_start,tau.hat,log_rolled_cases,shifted_log_rolled_cases,predicted.grf.future.last,beta_wsize=2,beta_wsize=3,beta_wsize=4
0,1001.0,Autauga,Alabama,2020-04-17,87.0,0.026873,3.030824,3.122994,3.218935,NaN,NaN,NaN
1,1001.0,Autauga,Alabama,2020-04-18,88.0,0.010264,3.030824,3.160035,3.102674,0.000000,NaN,NaN
2,1001.0,Autauga,Alabama,2020-04-19,89.0,0.018760,3.044522,3.183989,3.175840,0.013699,0.006849,NaN
3,1001.0,Autauga,Alabama,2020-04-20,90.0,0.021026,3.064725,3.213145,3.211908,0.020203,0.016951,0.011540
4,1001.0,Autauga,Alabama,2020-04-21,91.0,0.012541,3.064725,3.241476,3.152509,0.000000,0.010101,0.012191
...,...,...,...,...,...,...,...,...,...,...,...,...
177567,99999.0,New York City,New York,2022-12-20,1064.0,0.019618,11.325935,11.308842,11.463264,0.020137,0.024722,0.026704
177568,99999.0,New York City,New York,2022-12-21,1065.0,0.017220,11.341732,11.300346,11.462273,0.015797,0.017967,0.021586
177569,99999.0,New York City,New York,2022-12-22,1066.0,0.014948,11.354044,11.290173,11.458682,0.012312,0.014054,0.016054
177570,99999.0,New York City,New York,2022-12-23,1067.0,0.009073,11.356125,11.281184,11.419639,0.002081,0.007196,0.010288
